### Bibliotecas e Dependências

In [ ]:
pip install -U pymupdf4llm

In [ ]:
import pandas as pd
import requests
import pymupdf4llm
import re
import os
import json
import unicodedata

Leitura do Arquivo

In [ ]:
df = pd.read_csv(r'C:\Users\Thiago Lobo\Projeto-Mestrado\Corpus\metadados_completos_sol_sbc.csv')

In [ ]:
df["index"] = range(1, len(df) + 1)

In [ ]:
df.info()

In [ ]:
df_test = df.head(5)

In [ ]:
df_test.info()

In [ ]:
"""
1. Requisição para obtenção dos PDF - OK
2. Armazenamento do PDF Localmente - OK
3. Extração do conteúdo do PDF utilizando o pymupdf4llm armazenando o resultado em formato Markdown - OK
4. Identificação das seções e extração do conteúdo de cada seção - OK
5. Normalização do Texto 
6. Detecção do Idioma da Introdução ou Conclusão 
6. Armazenamento do conteúdo extraído em JSON (um arquivo JSON por artigo)
"""

In [ ]:
def save_article_json(article_data, output_path):

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            article_data,
            f,
            ensure_ascii=False,
            indent=4
        )

In [ ]:
def normalize_text(text):

    # normalização unicode
    text = unicodedata.normalize("NFKC", text)

    # remove hifenização de quebra de linha
    text = re.sub(r'-\s*\n\s*', '', text)

    # transforma quebras simples em espaço
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

    # remove múltiplos espaços
    text = re.sub(r'\s+', ' ', text)

    # remove espaços nas bordas
    text = text.strip()

    return text

In [ ]:
def extract_sections(md_text):
    pattern = re.compile(r'^(#{1,6})\s+(.*)$', re.MULTILINE)

    matches = list(pattern.finditer(md_text))

    sections = []

    for i, match in enumerate(matches):
        level = len(match.group(1))

        # limpa markdown do título
        title = match.group(2).strip()
        title = re.sub(r'\*+', '', title).strip()

        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(md_text)

        content = md_text[start:end].strip()

        content = normalize_text(content)

        sections.append({
            "level": level,
            "title": title,
            "content": content
        })

    return sections

In [ ]:
os.makedirs("pdfs", exist_ok=True)
os.makedirs("markdown", exist_ok=True)
os.makedirs("json", exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/pdf"
}

total = len(df_test)

for index, row in df_test.iterrows():

    paper_id = f"sbc_{index}"

    pdf_path = f"pdfs/{paper_id}.pdf"
    md_path = f"markdown/{paper_id}.md"
    json_path = f"json/{paper_id}.json"

    try:

        print(f"\n[{index + 1}/{total}] Processando {paper_id}")

        # evita reprocessamento
        if os.path.exists(json_path):
            print("Artigo já processado.")
            continue

        # =========================
        # URL DO PDF
        # =========================

        url = row["URL_Paper"]

        # troca /view/ por /download/
        pdf_url = url.replace("/view/", "/download/")

        print("URL ORIGINAL :", url)
        print("URL PDF      :", pdf_url)

        # =========================
        # DOWNLOAD DO PDF
        # =========================

        response = requests.get(
            pdf_url,
            headers=headers,
            timeout=60
        )

        response.raise_for_status()

        content_type = response.headers.get("Content-Type", "")

        print("CONTENT TYPE:", content_type)

        # valida conteúdo vazio
        if len(response.content) == 0:
            raise Exception("PDF vazio")

        # valida assinatura do arquivo
        if not response.content.startswith(b"%PDF"):
            raise Exception("Arquivo baixado não é PDF")

        # salva PDF
        with open(pdf_path, "wb") as f:
            f.write(response.content)

        print("PDF salvo com sucesso.")

        # =========================
        # PDF -> MARKDOWN
        # =========================

        md_text = pymupdf4llm.to_markdown(
            pdf_path,
            write_images=False
        )

        if not md_text.strip():
            raise Exception("Markdown vazio")

        # salva markdown
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(md_text)

        print("Markdown gerado com sucesso.")

        # =========================
        # EXTRAÇÃO DE SEÇÕES
        # =========================

        sections = extract_sections(md_text)

        if len(sections) == 0:
            raise Exception("Nenhuma seção encontrada")

        print(f"{len(sections)} seções encontradas.")

        # =========================
        # ESTRUTURA FINAL
        # =========================

        article_data = {
            "paper_id": paper_id,
            "title": row["Title"],
            "event": row["Event"],
            "authors": row["Authors"],
            "abstract_original": row["Abstract"],
            "url_paper": pdf_url,
            "sections": sections
        }

        # =========================
        # SALVA JSON
        # =========================

        save_article_json(article_data, json_path)

        print("JSON salvo com sucesso.")

        # =========================
        # DEBUG RESUMIDO
        # =========================

        for s in sections[:3]:

            print("=" * 50)
            print("TÍTULO:", s["title"])
            print("TAMANHO:", len(s["content"]))
            print(s["content"][:300])

    except Exception as e:

        print(f"Erro no artigo {paper_id}: {e}")